In [ ]:
# .env를 읽어오는 라이브러리 설치
# pip install python-dotenv

In [1]:
import os
import pandas as pd
import json
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List

In [5]:
# 1. .env 파일에 적힌 내용을 컴퓨터 환경 변수로 쫙 불러옵니다.
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

class QAPair(BaseModel):
    question: str
    reference: str
    keyword_groups: List[List[str]]

class QAList(BaseModel):
    qa_pairs: List[QAPair]

def generate_factoid_qa(row):
    qa_list = []
    doc_id = str(row['공고 번호'])
    title = str(row['사업명'])
    agency = str(row['발주 기관'])
    
    if pd.notna(row['사업 금액']):
        budget = int(row['사업 금액'])
        qa_list.append({
            "qid": f"{doc_id}_fact_budget",
            "question": f"{agency}에서 발주한 '{title}' 사업의 배정 예산은 얼마인가요?",
            "gold_ids": [doc_id],
            "reference": f"{budget:,}원 입니다.",
            "gold_keyword_groups": [[str(budget)]],
            "required_keyword_groups": [[str(budget)]]
        })
        
    if pd.notna(row['입찰 참여 마감일']):
        deadline = str(row['입찰 참여 마감일'])
        date_only = deadline.split(" ")[0] 
        qa_list.append({
            "qid": f"{doc_id}_fact_deadline",
            "question": f"'{title}' 사업의 입찰 참여 마감일과 시간은 언제인가요?",
            "gold_ids": [doc_id],
            "reference": deadline,
            "gold_keyword_groups": [[date_only]],
            "required_keyword_groups": [[date_only]]
        })
    return qa_list

def generate_llm_qa(row, num_questions=2):
    doc_id = str(row['공고 번호'])
    context = f"[사업 요약]\n{row['사업 요약']}\n\n[제안요청서 본문 일부]\n{str(row['텍스트'])[:1500]}"
    
    prompt = f"""
    당신은 RAG 시스템 평가용 Q&A 데이터셋 생성 전문가입니다.
    다음 사업 공고 문서를 읽고, RAG 모델이 문맥을 이해해야 풀 수 있는 질문 {num_questions}개를 만들어주세요.
    
    [규칙]
    1. 'question'은 사용자가 검색창에 입력할 자연스러운 문장.
    2. 'reference'는 문서에 기반한 정확한 모범 답안.
    3. 'keyword_groups'는 정답 인정에 필수적인 핵심 키워드 그룹 (동의어 포함).
    
    [문서 데이터]
    {context}
    """
    try:
        print(f"      -> 🤖 LLM (gpt-5-mini) API 호출 중... (대기 중)") # ⬅️ 진행 상황 확인용
        response = client.beta.chat.completions.parse(
            model="gpt-5-mini-2025-08-07", 
            messages=[
                {"role": "system", "content": "You are a helpful assistant designed to output JSON."},
                {"role": "user", "content": prompt}
            ],
            response_format=QAList,
            timeout=30 # 🌟 [중요] 30초 이상 응답이 없으면 무한 대기하지 않고 에러를 발생시킴
        )
        print(f"      -> ✅ LLM API 응답 완료!") # ⬅️ 진행 상황 확인용
        
        results = []
        for i, qa in enumerate(response.choices[0].message.parsed.qa_pairs):
            results.append({
                "qid": f"{doc_id}_llm_{i+1}",
                "question": qa.question,
                "gold_ids": [doc_id],
                "reference": qa.reference,
                "gold_keyword_groups": qa.keyword_groups,
                "required_keyword_groups": qa.keyword_groups
            })
        return results
    except Exception as e:
        print(f"      -> ❌ LLM QA 생성 에러 (ID: {doc_id}): {e}")
        return []

    
def create_eval_dataset(csv_path, output_json_path):
    print(f"📁 '{csv_path}' 읽는 중...")
    df = pd.read_csv(csv_path)
    final_eval_items = []
    
    print(f"🚀 전체 데이터 {len(df)}건에 대한 모의고사 출제를 시작합니다!")
    
    # 🌟 [수정 완료] head(2)를 지우고 전체 데이터(df)를 순회하도록 변경했습니다.
    for idx, row in df.iterrows(): 
        print(f"\n⏳ 처리 중... [{idx+1}/{len(df)}] 사업명: {str(row['사업명'])[:20]}...")
        
        # 1. 팩트 기반 규칙 생성 (비용 무료, 즉시 생성)
        final_eval_items.extend(generate_factoid_qa(row))
        
        # 2. LLM 기반 심층 QA 생성 (API 호출)
        print("      -> 🧠 LLM 기반 심층 QA 창작 중...")
        final_eval_items.extend(generate_llm_qa(row, num_questions=2))
            
    # 저장 폴더가 없으면 만들고, JSON 파일로 예쁘게 저장
    os.makedirs(os.path.dirname(output_json_path), exist_ok=True)
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_eval_items, f, ensure_ascii=False, indent=4)
        
    print(f"\n✅ 전체 데이터셋 생성 완료! 총 {len(final_eval_items)}개의 QA 세트가 '{output_json_path}'에 저장되었습니다.")

In [6]:
# 실행
if __name__ == "__main__":
    csv_file_path = '../data/raw/data_list.csv'
    output_eval_path = '../data/processed/eval/eval_dataset.json'
    
    if os.path.exists(csv_file_path):
        create_eval_dataset(csv_path=csv_file_path, output_json_path=output_eval_path)
    else:
        print(f"❌ '{csv_file_path}' 파일을 찾을 수 없습니다.")

📁 '../data/raw/data_list.csv' 읽는 중...
🚀 전체 데이터 100건에 대한 모의고사 출제를 시작합니다!

⏳ 처리 중... [1/100] 사업명: 한영대학교 특성화 맞춤형 교육환경 구...
      -> 🧠 LLM 기반 심층 QA 창작 중...
      -> 🤖 LLM (gpt-5-mini) API 호출 중... (대기 중)
      -> ✅ LLM API 응답 완료!

⏳ 처리 중... [2/100] 사업명: 2024년 대학산학협력활동 실태조사 ...
      -> 🧠 LLM 기반 심층 QA 창작 중...
      -> 🤖 LLM (gpt-5-mini) API 호출 중... (대기 중)
      -> ✅ LLM API 응답 완료!

⏳ 처리 중... [3/100] 사업명: EIP3.0 고압가스 안전관리 시스템...
      -> 🧠 LLM 기반 심층 QA 창작 중...
      -> 🤖 LLM (gpt-5-mini) API 호출 중... (대기 중)
      -> ✅ LLM API 응답 완료!

⏳ 처리 중... [4/100] 사업명: 도시계획위원회 통합관리시스템 구축용역...
      -> 🧠 LLM 기반 심층 QA 창작 중...
      -> 🤖 LLM (gpt-5-mini) API 호출 중... (대기 중)
      -> ✅ LLM API 응답 완료!

⏳ 처리 중... [5/100] 사업명: 봉화군 재난통합관리시스템 고도화 사업...
      -> 🧠 LLM 기반 심층 QA 창작 중...
      -> 🤖 LLM (gpt-5-mini) API 호출 중... (대기 중)
      -> ✅ LLM API 응답 완료!

⏳ 처리 중... [6/100] 사업명: 전기안전 관제시스템 보안 모듈 개발 ...
      -> 🧠 LLM 기반 심층 QA 창작 중...
      -> 🤖 LLM (gpt-5-mini) API 호출 중... (대기 중)
      -> ✅ LLM API 응답 완료!

⏳ 처리 중...